# NEOPAX Profile Closure

## What you will learn
Why profiles and sources close the loop between field optimization and performance.

## Codes used
NEOPAX in research mode; educational radial diffusion/source model by default.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`10_profile_evolution.png` and `10_power_balance.png`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Learning frame

This notebook is a deliberately small project: define one metric, produce one plot, expose one failure mode, and identify where a real code would enter.

In [ ]:
from sos2026.transport_helpers import profile_closure
from sos2026.plotting import savefig, caption

## 2. Load or generate the teaching data

Cached mode uses small arrays so the conceptual workflow is always available.

In [ ]:
d = profile_closure()
pd.DataFrame({"r": d["r"], "source": d["source"], "chi_base": d["chi_base"]}).head()

## 3. Make the primary plot

Every plot has a one-sentence caption because students should know how to read it without guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.7))
ax.plot(d["r"], d["Ti_base"], lw=2, label="Ti baseline")
ax.plot(d["r"], d["Ti_turbulent"], lw=2, label="Ti with extra turbulent chi")
ax.plot(d["r"], d["Te"], lw=2, label="Te")
ax.set_xlabel("r/a")
ax.set_ylabel("temperature proxy")
ax.set_title("Profile closure fallback")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
caption(ax, "The field design is not complete until profiles and sources close the transport loop.")
savefig(fig, "10_profile_evolution.png")
plt.show()

## 4. Probe the metric

A metric becomes useful for optimization only when we understand how it changes across design choices.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.plot(d["r"], d["source"], lw=2, label="source")
ax.plot(d["r"], d["flux_balance"], lw=2, label="residual")
ax.set_xlabel("r/a")
ax.set_ylabel("normalized power density")
ax.set_title("Power-balance check")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
caption(ax, "A residual plot teaches whether assumed transport closes with the chosen source.")
savefig(fig, "10_power_balance.png")
plt.show()

## 5. Interpret the design consequence

The table below translates the plot into an optimization decision.

In [ ]:
closure = pd.DataFrame({
    "loop item": ["geometry", "transport coefficients", "sources", "profiles", "updated objectives"],
    "question": ["what field?", "what flux?", "what heating/fueling?", "what gradients?", "what changes next?"],
})
closure

## 6. Failure mode

The cached plot is useful only if we say what it does not prove.

In [ ]:
failure_mode = pd.DataFrame({
    "cached_mode_proves": ["workflow shape", "plot grammar", "where the metric enters"],
    "cached_mode_does_not_prove": ["validated physics", "final design ranking", "runtime scalability"],
})
failure_mode

## 7. Research-mode hook

Run this cell only after timing the package on the lecture machine.

In [ ]:
if RUN_MODE == "research":
    import NEOPAX
    print("NEOPAX import OK; research path: inspect examples for radial transport closure.")
else:
    print("Cached mode: research package path skipped intentionally.")

## 8. Mini project handoff

Use this notebook during the lecture as the computational project slide points to: change one parameter, regenerate one plot, and explain one design tradeoff.

In [ ]:
project_steps = pd.DataFrame({
    "step": [1, 2, 3, 4],
    "action": ["identify metric", "change one input", "regenerate plot", "state failure mode"],
})
project_steps

## Try this
Change one scalar or one row in the cached data and regenerate the primary plot.

## Expected qualitative answer
The plot should move in a physically interpretable direction, but the cached result remains an educational proxy.

## Research extension
Replace the cached data source with the corresponding real package output after timing and API verification.